# Creative Floors - Automated Form Testing

Selenium tests for all 4 forms on creativefloors.co (local dev at `localhost/m`).

| Form | ID | Page(s) | Fields |
|---|---|---|---|
| **Compact** | `compactCtForm` | index.php, flooring-*.php | fullName, email, phone, project, consent |
| **Contact** | `contactForm` | pages/contact.php | firstName, lastName, email, phone, city, zip, project, sqft, comments |
| **FAQ** | `questionForm` | index.php, pages/services.php | question, zip, email |
| **Newsletter** | `newsletterForm` | footer (all pages) | name, zip, email |

**Flow:** Fill form -> JS validates -> reCAPTCHA token -> POST JSON to send-email.php -> redirect to thank-you.php

**Note:** Uses `X-Test-Mode: automation` header via CDP so send-email.php can route test emails to a separate inbox and lower the reCAPTCHA threshold. Requires the test-config.php and send-email.php modifications described in testing_forms.md.

In [1]:
# Setup - imports and shared helpers
!pip install webdriver_manager

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import Select
from selenium.common.exceptions import ElementNotInteractableException, InvalidElementStateException

import time

BASE_URL = "http://creativefloors.co/m"
THANK_YOU_PATH = "/m/pages/thank-you.php"
WAIT_TIMEOUT = 15

def create_driver():
    """Create a Chrome driver with X-Test-Mode header injected via CDP."""
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    return driver

def wait_for_redirect(driver, path=THANK_YOU_PATH, timeout=WAIT_TIMEOUT):
    """Wait until the browser redirects to the thank-you page."""
    WebDriverWait(driver, timeout).until(EC.url_contains(path))
    print(f"  -> Redirected to: {driver.current_url}")

def fill_and_submit(driver, fields, submit_id):
    """Fill form fields and click submit. fields = dict of {css_selector: value}."""
    for selector, value in fields.items():
        el = driver.find_element(By.CSS_SELECTOR, selector)
        if el.get_attribute("type") == "checkbox":
            if value and not el.is_selected():
                el.click()
        else:
            el.clear()
            el.send_keys(value)
    driver.find_element(By.ID, submit_id).click()

print("Setup complete.")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Setup complete.


In [4]:
# Test: Compact Form (compactCtForm) - Home page
# Fields: fullName, contactemail, phone, project, consent
# No ZIP field (hardcoded to 60002 by JS), no comments

driver = create_driver()
try:
    driver.get(f"{BASE_URL}/index.php")
    print("Compact Form - index.php")
    
    # Scroll to form so it's interactable
    form = WebDriverWait(driver, WAIT_TIMEOUT).until(
        EC.presence_of_element_located((By.ID, "compactCtForm"))
    )
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", form)
    time.sleep(1)

    def fill_and_submit(driver, field_values, submit_button_id):
        for selector, value in field_values.items():
            element = driver.find_element(By.CSS_SELECTOR, selector)
            
            if isinstance(value, tuple):
                # Special handling for dropdown (project)
                text_to_select, mode = value
                if mode == "select_by_text":
                    # Wait until clickable + force scroll
                    WebDriverWait(driver, 10).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, selector))
                    )
                    driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center', inline: 'center'});",
                        element
                    )
                    time.sleep(0.4)  # small grace period
                    
                    # Retry loop for stubborn dropdowns
                    success = False
                    for attempt in range(3):
                        try:
                            # Try to force visibility (helps with opacity/overlay issues)
                            driver.execute_script(
                                "arguments[0].style.opacity = '1'; "
                                "arguments[0].style.display = 'block'; "
                                "arguments[0].style.visibility = 'visible';",
                                element
                            )
                            sel = Select(element)
                            sel.select_by_visible_text(text_to_select)  # "Hardwood"
                            success = True
                            break
                        except (ElementNotInteractableException, InvalidElementStateException) as ex:
                            if attempt == 2:
                                raise Exception(f"Failed to select '{text_to_select}' after 3 attempts: {ex}")
                            time.sleep(0.8)
                    if not success:
                        raise Exception(f"Could not select '{text_to_select}' from dropdown")
                    
            elif selector == "#consent":
                # Checkbox
                if value and not element.is_selected():
                    element.click()
            else:
                # Normal text inputs
                element.clear()
                element.send_keys(value)
        
        # Submit
        driver.find_element(By.ID, submit_button_id).click()

    fill_and_submit(driver, {
        "#fullName": "Selenium Test User",
        "#contactemail": "selenium.test@creativefloors.co",
        "#phone": "630-555-0123",
        "#project": ("Hardwood", "select_by_text"),   # ← text must match exactly
        "#consent": True
    }, "submit-h")

    wait_for_redirect(driver)
    print(" PASSED")

except Exception as e:
    driver.save_screenshot("compact_form_error.png")
    print(f" FAILED: {e}")

finally:
    driver.quit()

Compact Form - index.php
  -> Redirected to: https://creativefloors.co/m/pages/thank-you.php
 PASSED


In [5]:
driver = create_driver()
try:
    driver.get(f"{BASE_URL}/pages/contact.php")
    print("Contact Form - pages/contact.php")
    
    form = WebDriverWait(driver, WAIT_TIMEOUT).until(
        EC.presence_of_element_located((By.ID, "contactForm"))
    )
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", form)
    time.sleep(1)

    def fill_and_submit(driver, field_values, submit_button_id):
        for selector, value in field_values.items():
            element = driver.find_element(By.CSS_SELECTOR, selector)
            
            if isinstance(value, tuple):
                # Special handling for dropdown (project)
                text_to_select, mode = value
                if mode == "select_by_text":
                    # Wait until clickable + force scroll
                    WebDriverWait(driver, 10).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, selector))
                    )
                    driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center', inline: 'center'});",
                        element
                    )
                    time.sleep(0.4)  # small grace period after scroll
                    
                    # Retry loop for visibility/interactability issues
                    success = False
                    for attempt in range(3):
                        try:
                            # Force visibility (helps if overlaid/hidden/opacity low)
                            driver.execute_script(
                                "arguments[0].style.opacity = '1'; "
                                "arguments[0].style.display = 'block'; "
                                "arguments[0].style.visibility = 'visible';",
                                element
                            )
                            sel = Select(element)
                            sel.select_by_visible_text(text_to_select)  # e.g. "Carpet"
                            success = True
                            break
                        except (ElementNotInteractableException, InvalidElementStateException) as ex:
                            if attempt == 2:
                                raise Exception(f"Failed to select '{text_to_select}' after 3 attempts: {ex}")
                            time.sleep(0.8)
                    if not success:
                        raise Exception(f"Could not select '{text_to_select}' from dropdown")
                    
            elif selector in ("#subscribe", "#dataConsent"):
                if value and not element.is_selected():
                    element.click()
            else:
                # Text inputs / textarea
                element.clear()
                element.send_keys(value)
        
        # Submit
        driver.find_element(By.ID, submit_button_id).click()

    fill_and_submit(driver, {
        "#firstName": "Selenium",
        "#lastName": "Tester",
        "#contactemail": "selenium.test@creativefloors.co",
        "#phone": "630-555-0456",
        "#location": "Aurora",
        "#zip": "60502",
        "#project": ("Carpet", "select_by_text"),   # text must match exactly as shown in dropdown
        "#sqft": "500",
        "#subscribe": True,
        "#dataConsent": True,
        "#comments": "This is an automated test submission. Please ignore."
    }, "submit-c")
    time.sleep(2)
    wait_for_redirect(driver)
    print(" PASSED")

except Exception as e:
    driver.save_screenshot("contact_form_error.png")
    print(f" FAILED: {e}")

finally:
    driver.quit()

Contact Form - pages/contact.php
  -> Redirected to: https://creativefloors.co/m/pages/thank-you.php
 PASSED


In [5]:
# Test: FAQ Form (questionForm) - Home page (FAQ section, "Ask a question" tab)
# Fields: question, zip, email
# No name, no phone, no comments, no honeypot

driver = create_driver()
try:
    driver.get(f"{BASE_URL}/index.php")
    print("FAQ Form - index.php")

    # Scroll the FAQ heading into view first
    faq_heading = WebDriverWait(driver, WAIT_TIMEOUT).until(
        EC.presence_of_element_located((By.ID, "faq"))
    )
    driver.execute_script("arguments[0].scrollIntoView({block: 'start'});", faq_heading)
    time.sleep(1)

    # Click the "Ask" pill tab via JS to avoid interception
    ask_tab = driver.find_element(By.CSS_SELECTOR, "#faq ~ .tab-class a[href='#tab-2']")
    driver.execute_script("arguments[0].click();", ask_tab)

    # Wait for the form to be visible, then scroll it into view
    form = WebDriverWait(driver, WAIT_TIMEOUT).until(
        EC.visibility_of_element_located((By.ID, "questionForm"))
    )
    driver.execute_script("arguments[0].scrollIntoView({block: 'start'});", form)

    # Fill fields
    for selector, value in {
        "#question":           "What hardwood flooring options do you offer? - Automated Test",
        "#questionForm #zip":  "60502",
        "#questionForm #email": "selenium.test@creativefloors.co"
    }.items():
        el = driver.find_element(By.CSS_SELECTOR, selector)
        el.clear()
        el.send_keys(value)

    # Scroll to submit and click via JS
    submit_btn = driver.find_element(By.ID, "submit-q")
    driver.execute_script("arguments[0].scrollIntoView({block: 'start'});", submit_btn)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", submit_btn)

    wait_for_redirect(driver)
    print("  PASSED")
except Exception as e:
    driver.save_screenshot("faq_form_error.png")
    print(f"  FAILED: {e}")
finally:
    driver.quit()

FAQ Form - index.php
  -> Redirected to: https://creativefloors.co/m/pages/thank-you.php
  PASSED


In [6]:
# Test: Newsletter Form (newsletterForm) - Footer (present on all pages, testing from index)
# Fields: newsletterName, zip, newsletterEmail
# Has honeypot (middleName), no phone, no comments
# Single name field (not first/last)

driver = create_driver()
try:
    driver.get(f"{BASE_URL}/index.php")
    print("Newsletter Form - index.php (footer)")

    # Scroll to footer where newsletter form lives
    form = WebDriverWait(driver, WAIT_TIMEOUT).until(
        EC.presence_of_element_located((By.ID, "newsletterForm"))
    )
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", form)
    time.sleep(1)

    fill_and_submit(driver, {
        "#newsletterName":        "Selenium Newsletter Test",
        "#newsletterForm #zip":   "60502",
        "#newsletterEmail":       "selenium.test@creativefloors.co"
    }, "submit-n")

    wait_for_redirect(driver)
    print("  PASSED")
except Exception as e:
    driver.save_screenshot("newsletter_form_error.png")
    print(f"  FAILED: {e}")
finally:
    driver.quit()

Newsletter Form - index.php (footer)
  -> Redirected to: https://creativefloors.co/m/pages/thank-you.php
  PASSED
